<a href="https://colab.research.google.com/github/Ademola-Olorunnisola/Ademola-Olorunnisola.github.io/blob/main/Players_History_Last_Four_Seasons.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
Downloads gameweek-by-gameweek FPL data for the last N completed seasons
from the vaastav/Fantasy-Premier-League GitHub repo.

Each season's merged_gw.csv contains one row per player per gameweek,
with points, minutes, goals, assists, opponent, etc. already merged.
"""

import pandas as pd
import requests
from pathlib import Path

BASE_URL = "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data"

# Last 4 completed seasons as of Aug 2026 (2026-27 hasn't started yet)
SEASONS = ["2022-23", "2023-24", "2024-25", "2025-26"]

# Uses the current working directory instead of __file__ so this also works
# inside a Jupyter notebook cell (where __file__ isn't defined).
# Run this notebook/script from inside the fpl-model/ folder, or adjust
# PROJECT_ROOT below to point at wherever you cloned/created the project.
PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)


def download_season(season: str) -> pd.DataFrame:
    """Download merged_gw.csv for one season and tag it with the season label."""
    url = f"{BASE_URL}/{season}/gws/merged_gw.csv"
    print(f"Downloading {season} from {url} ...")

    resp = requests.get(url, timeout=30)
    resp.raise_for_status()

    # Save the raw file locally so we don't have to re-download it every run
    local_path = RAW_DIR / f"merged_gw_{season}.csv"
    local_path.write_bytes(resp.content)

    # merged_gw.csv files are typically latin-1 encoded (accented player names)
    df = pd.read_csv(local_path, encoding="latin-1")
    df["season"] = season
    print(f"  -> {len(df)} rows, {df['name'].nunique() if 'name' in df.columns else '?'} unique players")
    return df


def main():
    frames = []
    for season in SEASONS:
        try:
            frames.append(download_season(season))
        except requests.HTTPError as e:
            print(f"  ! Failed to download {season}: {e}")

    all_seasons = pd.concat(frames, ignore_index=True)
    out_path = PROJECT_ROOT / "data" / "processed" / "historical_gws_raw.csv"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    all_seasons.to_csv(out_path, index=False)

    print(f"\nSaved combined dataset: {out_path}")
    print(f"Total rows: {len(all_seasons)}")
    print(f"Columns: {list(all_seasons.columns)}")


if __name__ == "__main__":
    main()

  -> 26505 rows, 777 unique players
  -> 29725 rows, 869 unique players
  -> 27605 rows, 805 unique players
  -> 29757 rows, 841 unique players

Saved combined dataset: /content/data/processed/historical_gws_raw.csv
Total rows: 113592
Columns: ['name', 'position', 'team', 'xP', 'assists', 'bonus', 'bps', 'clean_sheets', 'creativity', 'element', 'expected_assists', 'expected_goal_involvements', 'expected_goals', 'expected_goals_conceded', 'fixture', 'goals_conceded', 'goals_scored', 'ict_index', 'influence', 'kickoff_time', 'minutes', 'opponent_team', 'own_goals', 'penalties_missed', 'penalties_saved', 'red_cards', 'round', 'saves', 'selected', 'starts', 'team_a_score', 'team_h_score', 'threat', 'total_points', 'transfers_balance', 'transfers_in', 'transfers_out', 'value', 'was_home', 'yellow_cards', 'GW', 'season', 'mng_clean_sheets', 'mng_draw', 'mng_goals_scored', 'mng_loss', 'mng_underdog_draw', 'mng_underdog_win', 'mng_win', 'modified', 'clearances_blocks_interceptions', 'defensive

In [3]:
"""
Pulls current-season data from the official FPL API.

NOTE: run this on your own machine — fantasy.premierleague.com isn't reachable
from this sandbox's network, so it can't be tested here.

Endpoints used:
- bootstrap-static: current player list, prices, injury/availability status, ICT index
- fixtures: full fixture list with FPL's own difficulty ratings
- element-summary/{id}: per-player match history for the current season so far
"""

import requests
import pandas as pd
from pathlib import Path
import time

BASE = "https://fantasy.premierleague.com/api"
# Uses cwd instead of __file__ so this also works inside a Jupyter notebook cell.
# Run this from inside the fpl-model/ folder.
PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)


def get_bootstrap():
    r = requests.get(f"{BASE}/bootstrap-static/", timeout=30)
    r.raise_for_status()
    return r.json()


def get_fixtures():
    r = requests.get(f"{BASE}/fixtures/", timeout=30)
    r.raise_for_status()
    return r.json()


def get_player_history(player_id: int):
    r = requests.get(f"{BASE}/element-summary/{player_id}/", timeout=30)
    r.raise_for_status()
    return r.json()


def main():
    print("Pulling bootstrap-static (players, teams, injury status)...")
    bootstrap = get_bootstrap()

    players = pd.DataFrame(bootstrap["elements"])
    teams = pd.DataFrame(bootstrap["teams"])
    positions = pd.DataFrame(bootstrap["element_types"])

    players.to_csv(RAW_DIR / "players_current.csv", index=False)
    teams.to_csv(RAW_DIR / "teams_current.csv", index=False)
    positions.to_csv(RAW_DIR / "positions_current.csv", index=False)
    print(f"  -> {len(players)} players, {len(teams)} teams")

    # Key columns to sanity check the pull worked:
    # status: 'a' = available, 'i' = injured, 'd' = doubtful, 's' = suspended
    # news: free-text injury/team news
    # chance_of_playing_next_round: 0-100, or null if fully fit
    print(players[["web_name", "status", "news", "chance_of_playing_next_round"]].head())

    print("\nPulling fixtures...")
    fixtures = pd.DataFrame(get_fixtures())
    fixtures.to_csv(RAW_DIR / "fixtures_current.csv", index=False)
    print(f"  -> {len(fixtures)} fixtures")

    # Per-player gameweek history for the current season so far.
    # This is a call per player, so we rate-limit lightly to be a good citizen.
    print("\nPulling per-player gameweek history for current season...")
    all_history = []
    for i, player_id in enumerate(players["id"]):
        try:
            data = get_player_history(player_id)
            hist = pd.DataFrame(data["history"])
            hist["player_id"] = player_id
            all_history.append(hist)
        except requests.HTTPError as e:
            print(f"  ! Failed for player {player_id}: {e}")
        if i % 50 == 0:
            print(f"  ...{i}/{len(players)} players done")
        time.sleep(0.05)  # small delay to avoid hammering the API

    history_df = pd.concat(all_history, ignore_index=True) if all_history else pd.DataFrame()
    history_df.to_csv(RAW_DIR / "current_season_gws.csv", index=False)
    print(f"\nSaved current-season gameweek history: {len(history_df)} rows")


if __name__ == "__main__":
    main()

Pulling bootstrap-static (players, teams, injury status)...
  -> 572 players, 20 teams
       web_name status                                 news  \
0          Raya      a                                        
1  Arrizabalaga      a                                        
2       Meslier      a                                        
3       Gabriel      a                                        
4      J.Timber      i  Groin injury - Expected back 21 Aug   

   chance_of_playing_next_round  
0                           NaN  
1                           NaN  
2                           NaN  
3                           NaN  
4                           0.0  

Pulling fixtures...
  -> 380 fixtures

Pulling per-player gameweek history for current season...
  ...0/572 players done
  ...50/572 players done
  ...100/572 players done
  ...150/572 players done
  ...200/572 players done
  ...250/572 players done
  ...300/572 players done
  ...350/572 players done
  ...400/572 players done
 

In [4]:
"""
Pulls current-season data from the official FPL API.

NOTE: run this on your own machine — fantasy.premierleague.com isn't reachable
from this sandbox's network, so it can't be tested here.

Endpoints used:
- bootstrap-static: current player list, prices, injury/availability status, ICT index
- fixtures: full fixture list with FPL's own difficulty ratings
- element-summary/{id}: per-player match history for the current season so far
"""

import requests
import pandas as pd
from pathlib import Path
import time

BASE = "https://fantasy.premierleague.com/api"
# Uses cwd instead of __file__ so this also works inside a Jupyter notebook cell.
# Run this from inside the fpl-model/ folder.
PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)


def get_bootstrap():
    r = requests.get(f"{BASE}/bootstrap-static/", timeout=30)
    r.raise_for_status()
    return r.json()


def get_fixtures():
    r = requests.get(f"{BASE}/fixtures/", timeout=30)
    r.raise_for_status()
    return r.json()


def get_player_history(player_id: int):
    r = requests.get(f"{BASE}/element-summary/{player_id}/", timeout=30)
    r.raise_for_status()
    return r.json()


def main():
    print("Pulling bootstrap-static (players, teams, injury status)...")
    bootstrap = get_bootstrap()

    players = pd.DataFrame(bootstrap["elements"])
    teams = pd.DataFrame(bootstrap["teams"])
    positions = pd.DataFrame(bootstrap["element_types"])

    players.to_csv(RAW_DIR / "players_current.csv", index=False)
    teams.to_csv(RAW_DIR / "teams_current.csv", index=False)
    positions.to_csv(RAW_DIR / "positions_current.csv", index=False)
    print(f"  -> {len(players)} players, {len(teams)} teams")

    # Key columns to sanity check the pull worked:
    # status: 'a' = available, 'i' = injured, 'd' = doubtful, 's' = suspended
    # news: free-text injury/team news
    # chance_of_playing_next_round: 0-100, or null if fully fit
    print(players[["web_name", "status", "news", "chance_of_playing_next_round"]].head())

    print("\nPulling fixtures...")
    fixtures = pd.DataFrame(get_fixtures())
    fixtures.to_csv(RAW_DIR / "fixtures_current.csv", index=False)
    print(f"  -> {len(fixtures)} fixtures")

    # Per-player gameweek history for the current season so far.
    # This is a call per player, so we rate-limit lightly to be a good citizen.
    print("\nPulling per-player gameweek history for current season...")
    all_history = []
    for i, player_id in enumerate(players["id"]):
        try:
            data = get_player_history(player_id)
            hist = pd.DataFrame(data["history"])
            hist["player_id"] = player_id
            all_history.append(hist)
        except requests.HTTPError as e:
            print(f"  ! Failed for player {player_id}: {e}")
        if i % 50 == 0:
            print(f"  ...{i}/{len(players)} players done")
        time.sleep(0.05)  # small delay to avoid hammering the API

    history_df = pd.concat(all_history, ignore_index=True) if all_history else pd.DataFrame()
    history_df.to_csv(RAW_DIR / "current_season_gws.csv", index=False)
    print(f"\nSaved current-season gameweek history: {len(history_df)} rows")


if __name__ == "__main__":
    main()

Pulling bootstrap-static (players, teams, injury status)...
  -> 572 players, 20 teams
       web_name status                                 news  \
0          Raya      a                                        
1  Arrizabalaga      a                                        
2       Meslier      a                                        
3       Gabriel      a                                        
4      J.Timber      i  Groin injury - Expected back 21 Aug   

   chance_of_playing_next_round  
0                           NaN  
1                           NaN  
2                           NaN  
3                           NaN  
4                           0.0  

Pulling fixtures...
  -> 380 fixtures

Pulling per-player gameweek history for current season...
  ...0/572 players done
  ...50/572 players done
  ...100/572 players done
  ...150/572 players done
  ...200/572 players done
  ...250/572 players done
  ...300/572 players done
  ...350/572 players done
  ...400/572 players done
 

In [5]:
url = "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/2025-26/gws/merged_gw.csv"
df = pd.read_csv(url, encoding="latin-1")
df.to_csv("historical_gws_2025_26.csv", index=False)

df.head()

,name,position,team,xP,assists,bonus,bps,clean_sheets,creativity,element,...,transfers_in,transfers_out,value,was_home,yellow_cards,clearances_blocks_interceptions,defensive_contribution,recoveries,tackles,GW
0,Reinildo Mandava,DEF,Sunderland,0.5,0,0,27,1,2.5,541,...,0,0,40,True,0,6,8,3,2,1
1,Lewis Dobbin,MID,Aston Villa,1.0,0,0,0,0,0.0,57,...,0,0,50,True,0,0,0,0,0,1
2,Ryan Christie,MID,Bournemouth,0.0,0,0,0,0,0.0,87,...,0,0,50,False,0,0,0,0,0,1
3,Zeki Amdouni,FWD,Burnley,0.0,0,0,0,0,0.0,216,...,0,0,50,False,0,0,0,0,0,1
4,Lucas Tolentino Coelho de Lima,MID,West Ham,2.6,0,0,11,0,14.2,612,...,0,0,60,False,0,0,6,5,1,1


In [6]:
import pandas as pd

BASE_URL = "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data"
SEASONS = ["2022-23", "2023-24", "2024-25", "2025-26"]

frames = []
for season in SEASONS:
    url = f"{BASE_URL}/{season}/gws/merged_gw.csv"
    print(f"Downloading {season}...")
    df = pd.read_csv(url, encoding="latin-1")
    df["season"] = season
    frames.append(df)

all_seasons = pd.concat(frames, ignore_index=True)
all_seasons.to_csv("historical_gws_raw.csv", index=False)
print(f"Saved {len(all_seasons)} rows across {len(SEASONS)} seasons")

Saved 113592 rows across 4 seasons
